### Load relevant libraries

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
from sqlalchemy import URL, create_engine, text, inspect

### 1. Database connection configuration
Set `PGUSER`, `PGPASSWORD`, `PGHOST`, and `PGPORT` in your environment as needed. `PGDATABASE` defaults to `ncaa_march_madness`. The account needs permission to create the database if it does not exist.


In [ ]:
# Read connection settings from the environment; never commit a password.
DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = int(os.environ.get("PGPORT", "5432"))
DB_NAME = os.environ.get("PGDATABASE", "ncaa_march_madness")

if not DB_PASSWORD:
    raise ValueError("Set PGPASSWORD in your environment before running this notebook.")

In [ ]:
admin_url = URL.create(
    "postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
    host=DB_HOST, port=DB_PORT, database="postgres",
)
admin_engine = create_engine(admin_url, isolation_level="AUTOCOMMIT")

with admin_engine.connect() as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :db_name"),
        {"db_name": DB_NAME},
    ).scalar()
    if not exists:
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", DB_NAME):
            raise ValueError("PGDATABASE must be a simple PostgreSQL identifier")
        conn.execute(text(f'CREATE DATABASE "{DB_NAME}"'))
        print(f"Created database: {DB_NAME}")
    else:
        print(f"Database already exists: {DB_NAME}")

In [ ]:
# Create the SQLAlchemy engine for the target database.
database_url = URL.create(
    "postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
    host=DB_HOST, port=DB_PORT, database=DB_NAME,
)
engine = create_engine(database_url)

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print("Connected to:", result.scalar())

### 2. Read in the data

In [ ]:
# Launch Jupyter from the repository root or its notebooks/ directory.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data" / "raw").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "data" / "raw").is_dir():
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ and provide data/raw/.")
RAW_DATA = REPO_ROOT / "data" / "raw"

In [ ]:
def clean_table_name(filename):
    """
    Converts file names into clean, Postgres-friendly table names.
    Example: 'MRegularSeasonCompactResults.csv' -> 'm_regular_season_compact_results'
    """
# Remove extension
    name = os.path.splitext(filename)[0]

    # Separate leading M/W prefix
    # MNCAATourney -> M_NCAATourney
    # MConference -> M_Conference
    if name.startswith(('M', 'W')) and len(name) > 1:
        name = name[0] + '_' + name[1:]

    # Handle acronym -> normal word boundary
    # NCAATourney -> NCAA_Tourney
    name = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', name)

    # Handle lowercase/digit -> uppercase boundary
    # RegularSeason -> Regular_Season
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Convert to lowercase
    name = name.lower()

    # Replace any sequence of non-alphanumeric characters with underscore
    name = re.sub(r'[^a-z0-9]+', '_', name)

    # Remove leading/trailing underscores
    name = name.strip('_')

    return name

### 3. Export to PostgreSQL

In [ ]:
# Export only new files to database
def export_csv_to_postgres():
    if not RAW_DATA.exists():
        print(f"Error: Directory {RAW_DATA} does not exist.")
        return

    # Get tables that already exist in PostgreSQL
    inspector = inspect(engine)
    existing_tables = set(inspector.get_table_names(schema="public"))

    # Loop through all CSV files in the directory
    for file_path in sorted(RAW_DATA.iterdir()):
        if file_path.is_file() and file_path.suffix.lower() == ".csv":
            file_name = file_path.name
            table_name = clean_table_name(file_name)

            # Skip files already represented in the database
            if table_name in existing_tables:
                print(f"Skipping {file_name} — table already exists.")
                continue

            print(f"Processing {file_name} -> Table: {table_name}...")

            try:
                # Only read the CSV if it needs to be imported
                df = pd.read_csv(file_path, low_memory=False)

                df.to_sql(
                    name=table_name,
                    schema="public",
                    con=engine,
                    if_exists="fail",
                    index=False,
                    chunksize=10000,
                )

                print(
                    f"Successfully exported {table_name} ({len(df)} rows)."
                )

                # Keep our local set current during this run
                existing_tables.add(table_name)

            except Exception as e:
                raise RuntimeError(f"Failed to export {file_name}") from e

In [ ]:
export_csv_to_postgres()